In [1]:
from beir.datasets.data_loader import GenericDataLoader
from beir import util

import spacy
from rank_bm25 import BM25Okapi

from sentence_transformers import SentenceTransformer

from tqdm.notebook import tqdm as tqdm_notebook
from tqdm import tqdm

from concurrent.futures import ProcessPoolExecutor,as_completed
import multiprocessing

import pathlib, os

import pandas as pd
import numpy as np

/home/michele/.local/lib/python3.10/site-packages/beir/datasets/data_loader.py:2: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


Data Preparation

In [2]:
def data_preparation(dataset:str):
       
       # Download dataset and unzip the dataset
       url = "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{}.zip".format(dataset)
       out_dir = os.path.join(pathlib.Path(os.path.abspath('')), "datasets")
       data_path = util.download_and_unzip(url, out_dir)
       
       # train/test split
       d_train,q_train,qr_train=GenericDataLoader(data_path).load(split="train")
       _,q_test,qr_test=GenericDataLoader(data_path).load(split="test")
       
       # to pandas dataframe
       train={"documents":pd.DataFrame.from_dict(d_train,orient="index").drop(["title"], axis=1),
              "queries":pd.DataFrame.from_dict(q_train,orient="index",columns=["text"]),
              "real":qr_train}
       
       test={"queries":pd.DataFrame.from_dict(q_test,orient="index",columns=["text"]),
              "real":qr_test}
              
              
       def data_tokenizer(train:dict[pd.DataFrame],test:dict[pd.DataFrame]):
              nlp = spacy.load("en_core_web_sm", disable=["tok2vec", "tagger", "parser", "attribute_ruler", "lemmatizer", "ner"])
              
              def tokenizer(text):
                     return [token.text for token in nlp(text) if not token.is_stop and not token.is_punct]
              
              def cleaning(tokenization):
                     return " ".join(tokenization)
              
              tqdm_notebook.pandas(desc="Document tokenization")
              train["documents"]["tokenized_text"]=train["documents"]["text"].progress_apply(tokenizer)
              
              tqdm_notebook.pandas(desc="Document cleaning")
              train["documents"]["text"]=train["documents"]["tokenized_text"].progress_apply(cleaning)
              
              tqdm_notebook.pandas(desc="Training queries tokenization")
              train["queries"]["tokenized_text"]=train["queries"]["text"].progress_apply(tokenizer)
              
              tqdm_notebook.pandas(desc="Training queries cleaning")
              train["queries"]["text"]=train["queries"]["tokenized_text"].progress_apply(cleaning)
              
              tqdm_notebook.pandas(desc="Test queries tokenization")
              test["queries"]["tokenized_text"]=test["queries"]["text"].progress_apply(tokenizer)
              
              tqdm_notebook.pandas(desc="Test queries cleaning")
              test["queries"]["text"]=test["queries"]["tokenized_text"].progress_apply(cleaning)
       
       #print(train["documents"].equals(test["documents"]))
       #print(train["queries"].equals(test["queries"]))
       
       #data tokenization using spacy
       data_tokenizer(train,test)
       
       return train,test

train,test=data_preparation("fiqa")

  0%|          | 0/57638 [00:00<?, ?it/s]

  0%|          | 0/57638 [00:00<?, ?it/s]

Document tokenization:   0%|          | 0/57638 [00:00<?, ?it/s]

Document cleaning:   0%|          | 0/57638 [00:00<?, ?it/s]

Training queries tokenization:   0%|          | 0/5500 [00:00<?, ?it/s]

Training queries cleaning:   0%|          | 0/5500 [00:00<?, ?it/s]

Test queries tokenization:   0%|          | 0/648 [00:00<?, ?it/s]

Test queries cleaning:   0%|          | 0/648 [00:00<?, ?it/s]

Sparse Vector Score

In [5]:
bar_queue = multiprocessing.Queue()

def BM25(bm25,start,stop):
    
    scores_per_query={}

    for index,row in train["queries"].iloc[start:stop].iterrows():
        
        scores=bm25.get_scores(row["text"])
        
        scores_per_query[index]=scores
        
        bar_queue.put_nowait(1)

    return scores_per_query

def BM25_Multiprocessing(n_jobs=8):
    
    scores_per_query_total={}
    
    def bar_handler(q):
        bar=tqdm(desc="Total", total=len(train["queries"]),bar_format="{l_bar}{bar}|{n_fmt}/{total_fmt} [{elapsed}]")
        
        while True:
            how_much=q.get()
            bar.update(how_much)
            bar.refresh()

    bar_process = multiprocessing.Process(target=bar_handler, args=(bar_queue, ), daemon=True)
    bar_process.start()

    with ProcessPoolExecutor(max_workers=n_jobs) as executor:
        
        bm25 = BM25Okapi(train["documents"]["text"].tolist())
        
        length=int(len(train["queries"])/n_jobs)
        futures=[]
            
        for start in range(0,len(train["queries"]),length):
            
            futures.append( executor.submit(BM25,bm25,start,start+length) )
        
        for future in as_completed(futures):
            result=future.result()
            scores_per_query_total.update(result)

    bar_process.terminate()
    
    return scores_per_query_total

Total:   3%|▎         |148/5500 [00:25]

In [ ]:
scores_per_query_sparse=BM25_Multiprocessing(8)

del bar_queue

In [20]:
giusto=0
k=100

for idx, res in scores_per_query_sparse.items():
    
    indexes = np.argpartition(res, -k)[-k:]  # Indices not sorted
    
    predicted_most_relevant=train["documents"].iloc[indexes[np.argsort(res[indexes])][::-1]].index.values
    
    #print(f'Predicted - Best document for query {idx} is the {train["documents"].iloc[winner].name}')
    #print(f'Ground Truth - Best document for query {idx} is the {train["real"][str(idx)]}\n')
    
    true_relevant=list(train["real"][str(idx)].keys())
    
    #print(true_relevant,predicted_most_relevant)
    
    if len(set(predicted_most_relevant)-set(true_relevant))<len(predicted_most_relevant):
        giusto+=1
    
giusto/5500

NameError: name 'scores_per_query_sparse' is not defined

Dense Vector Score

In [25]:
def dense_score(device:str="cuda"):
    
    model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2',device=device)

    documents_embeddings = model.encode(train["documents"]["text"])

    queries_embeddings = model.encode(train["queries"]["text"])

    return np.dot(queries_embeddings,documents_embeddings.transpose())

In [26]:
scores_per_query_dense=dense_score()

In [27]:
giusto=0
k=100

for idx,res in enumerate( scores_per_query_dense ):
    
    indexes = np.argpartition(res, -k)[-k:]  # Indices not sorted
    
    predicted_most_relevant=train["documents"].iloc[indexes[np.argsort(res[indexes])][::-1]].index.values
    
    #print(f'Predicted - Best document for query {idx} is the {train["documents"].iloc[winner].name}')
    #print(f'Ground Truth - Best document for query {idx} is the {train["real"][str(idx)]}\n')
    
    true_relevant=list(train["real"][ train["queries"].index.values[idx] ].keys())
    
    #print(true_relevant,predicted_most_relevant)
    
    if len(set(predicted_most_relevant)-set(true_relevant))<len(predicted_most_relevant):
        giusto+=1
    
giusto/5500

0.7232727272727273